<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week01_gd_optimization/gd_capstone.ipynb)

# Gradient-Based Optimization: From Calculus to GD/SGD

**TAE Week 1 Capstone**

## 0. Scope &amp; Reproducibility Notes

### Objective Under Study
$$
f(x) \;=\; \left|\tfrac{1}{2}x^{3} - \tfrac{3}{2}x^{2}\right| + \tfrac{1}{2}x
$$

### Deliverables &amp; Acceptance Checklist

1. **Analytical Foundations**: Formal definitions of $f(x)$, exact piecewise derivative $f'(x)$, and subgradient selection $\partial f(3) = [-4, 5]$ at the kink.
2. **Gradient Verification**: Central-difference numerical check against exact analytical derivatives.
3. **Deterministic GD Experiments**: Parameter sweep across 3 initializations ($x_0 \in \{-1.0, 0.5, 2.0\}$) $\times$ 4 step sizes ($\eta \in \{0.05, 0.10, 0.15, 0.20\}$) over horizon $K=200$.
4. **Stochastic GD Experiments**: Benchmarks comparing constant step sizes vs. Robbins–Monro diminishing step schedules ($\eta_k = \frac{\eta_0}{1 + \gamma k}$).
5. **Reproducible Figures**: Empirical replications of function landscape, step geometry, trajectories, overshooting, SGD paths, and schedule comparisons.
6. **Quantitative Metrics**: Final gap $f(x_K)-f(x_\star)$, best-so-far gap, steps-to-tolerance, and paired-seed statistical tests across 20 independent runs.
7. **Theoretical Extensions**: Local quadratic stability analysis and empirical probing of late-stage GD limit cycles near the non-smooth kink.

&gt; **Honest Reporting Note:** Figures 6–9 and 11 in the handout are explicitly *schematic* ("illustrative rather than the output of a fixed numerical run"). All figures in this notebook represent **actual deterministic numerical executions** under fixed random seeds.


## 1. Setup

All packages are in the standard Colab image; the cell below is a no-op on Colab and a
safety net elsewhere. Runtime target for the full notebook: **< 2 minutes**.

In [ ]:
# Install-if-missing (no-op on Colab; kept for the "single-click" requirement)
import importlib.util, subprocess, sys
for pkg in ("numpy", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:  # pragma: no cover
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


In [ ]:
%config InlineBackend.figure_format = 'retina'
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') 


## 2. Hyperparameters & Seed (single source of truth)

All experiment knobs live here. 

In [ ]:
SEED = 1                                    # fixed random seed
rng = np.random.default_rng(SEED)

# --- Protocol (handout, Section 7) ---
X0_LIST   = [-1.0, 0.5, 2.0]                # initializations
ETA_GD    = [0.05, 0.10, 0.15, 0.20]        # GD constant step sizes
ETA0_SGD  = [0.2, 0.1]                      # SGD initial steps (diminishing schedule)
GAMMA_SGD = [0.02, 0.05]                    # SGD decay rates
K         = 200                             # iteration horizon
SIGMA     = 0.5                             # SGD noise std (E[eps]=0, E[eps^2]<=sigma^2)
TOL       = 1e-3                            # tolerance for steps-to-tolerance metric

# --- Demo-specific values from the handout ---
ETA_GEOM      = 0.2    # step-geometry figure (Fig. 3): x0=2, eta=0.2 -> x'=1.9
ETA_TRAJ      = 0.15   # three-trajectory figure (Fig. 4)
ETA_OVERSHOOT = 0.6    # oscillation demo (Fig. 5): x0=0.5


## 3. Problem Setup: $f$, $f'$, Subgradient, Analytic Minimizer

Write $g(x) = \tfrac12 x^3 - \tfrac32 x^2 = \tfrac12 x^2 (x-3)$, so $f(x) = |g(x)| + \tfrac12 x$.

**Piecewise form** (handout, Prop. 1). Since $g(x)\le 0$ for $x<3$ and $g(x)>0$ for $x>3$:

$$
f(x) =
\begin{cases}
-\tfrac12 x^3 + \tfrac32 x^2 + \tfrac12 x, & x < 3,\\[2pt]
\;\;\tfrac12 x^3 - \tfrac32 x^2 + \tfrac12 x, & x > 3.
\end{cases}
$$

$f$ is differentiable at $x=0$ (because $g(0)=g'(0)=0$; handout Lemma 1) with $f'(0)=\tfrac12$,
and non-differentiable only at the kink $x=3$, where the one-sided limits are
$\lim_{x\uparrow 3} f'(x) = -4$ and $\lim_{x\downarrow 3} f'(x) = 5$, hence the Clarke subdifferential is

$$
\partial f(3) = [-4,\, 5].
$$

**Derivative on each branch:**

$$
f'(x) =
\begin{cases}
-\tfrac32 x^2 + 3x + \tfrac12, & x < 3,\\[2pt]
\;\;\tfrac32 x^2 - 3x + \tfrac12, & x > 3.
\end{cases}
$$

**Stationary points & global minimizer** (handout, Thm. 1). On $(-\infty,3)$ the roots of
$f'$ are $x_- = 1 - \tfrac{2}{3}\sqrt{3}$ and $x_+ = 1 + \tfrac{2}{3}\sqrt{3}$; the second-derivative test gives
$x_- = 1-\tfrac{2}{3}\sqrt{3}\approx -0.1547$ as a strict local (and unique global) minimizer
and $x_+\approx 2.1547$ as a strict local maximizer. For $x>3$, $f'>0$. The global minimum value is

$$
f(x_\star) = \tfrac{3}{2} - \tfrac{8}{9}\sqrt{3} \;\approx\; -3.96\times 10^{-2}.
$$


In [ ]:
def f(x):
    """Objective f(x) = |0.5 x^3 - 1.5 x^2| + 0.5 x."""
    x = np.asarray(x, dtype=np.float64)
    return np.abs(0.5 * x**3 - 1.5 * x**2) + 0.5 * x

def fprime(x):
    """Piecewise derivative on smooth branches (defined for x != 3)."""
    x = np.asarray(x, dtype=np.float64)
    left  = -1.5 * x**2 + 3.0 * x + 0.5   # x < 3
    right =  1.5 * x**2 - 3.0 * x + 0.5   # x > 3
    return np.where(x < 3.0, left, right)

def subgrad(x):
    """Subgradient selection: f'(x) off the kink; smallest-norm element of
    [-4, 5] (i.e., 0) exactly at x = 3 (handout, Section 8 recommendation).
    Exact equality (not a tolerance band) so nearby smooth points keep their
    true derivative; x = 3 is then an exact fixed point of the update."""
    if x == 3.0:
        return 0.0                        # smallest-norm selection in [-4, 5]
    return float(fprime(x))

# Analytic constants (exact expressions)
X_STAR = 1.0 - (2.0 / 3.0) * np.sqrt(3.0)          # global minimizer
F_STAR = 3.0 / 2.0 - (8.0 / 9.0) * np.sqrt(3.0)    # global minimum value
X_PLUS = 1.0 + (2.0 / 3.0) * np.sqrt(3.0)          # local maximizer

print(f"x_star = {X_STAR:+.6f}   f(x_star) = {F_STAR:+.6f}")
print(f"consistency check: f(X_STAR) - F_STAR = {f(X_STAR) - F_STAR:+.2e}")
assert np.isclose(f(X_STAR), F_STAR), "analytic minimum value mismatch"


## 4. Gradient Check: Analytic vs Numeric

Central differences $\dfrac{f(x+h)-f(x-h)}{2h}$ against the analytic $f'$ on both smooth
branches, avoiding the kink. Expected agreement: $\mathcal{O}(h^2)$.

In [ ]:
h = 1e-6
xs_check = np.concatenate([
    np.linspace(-2.0, 2.8, 13),   # left branch (avoids kink)
    np.linspace(3.2, 4.0, 5),     # right branch
])
num  = (f(xs_check + h) - f(xs_check - h)) / (2.0 * h)
ana  = fprime(xs_check)
err  = np.abs(num - ana)
print(f"max |analytic - numeric| = {err.max():.3e}  (n = {len(xs_check)} points)")
assert err.max() < 1e-6, "gradient check failed"
print("gradient check PASSED")


## 5. Function and Derivative

**Sign pattern of $f'$:** negative on $(-\infty, x_-)$, positive on $(x_-, x_+)$,
**negative on $(x_+, 3)$**, positive on $(3, \infty)$.

**Local geometry.** The derivative is negative immediately to the left of $x=3$ and positive immediately to its right, so the kink is a strict local minimizer with $f(3)=3/2$. It is not the global minimizer. Under our zero-subgradient selection, deterministic GD initialized exactly at $x=3$ stays there.

**Interpretation.** The derivative signs determine the direction of each GD step: leftward where the derivative is positive and rightward where it is negative. Finite steps can cross stationary points or the kink, so the sign pattern alone does not determine discrete-time basins of attraction or guarantee trapping. Section 6.5 examines particular starts and a particular step size.

In [ ]:
xs = np.linspace(-1.0, 4.0, 1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(xs, f(xs), label="$f(x)$")
axes[0].axvline(3.0, ls="--", c="gray", label="kink $x=3$")
axes[0].plot([X_STAR], [F_STAR], "o", c="#e84855", label="$x_\\star$")
axes[0].set(title="Objective $f$", xlabel="$x$", ylabel="$f(x)$")
axes[0].legend()

mask_l, mask_r = xs < 3.0, xs > 3.0
axes[1].plot(xs[mask_l], fprime(xs[mask_l]), c="#1b998b", label="$f'(x)$ (branches)")
axes[1].plot(xs[mask_r], fprime(xs[mask_r]), c="#1b998b")
axes[1].axvline(3.0, ls="--", c="gray")
axes[1].axhline(0.0, lw=0.8, c="k")
axes[1].plot([X_STAR, X_PLUS], [0, 0], "o", c="#e84855",
             label=f"roots $x_-\\approx{X_STAR:.4f}$, $x_+\\approx{X_PLUS:.4f}$")
axes[1].set(title="Piecewise derivative $f'$", xlabel="$x$", ylabel="$f'(x)$")
axes[1].legend()
fig.tight_layout()
plt.show()


## 6. Gradient Descent

$$
x_{k+1} = x_k - \eta\, g_k, \qquad
g_k \in \begin{cases}\{f'(x_k)\}, & x_k \neq 3,\\ \partial f(3), & x_k = 3.\end{cases}
$$

For sufficiently small $\eta$ the map is a contraction near $x_\star$
(since $f''(x_\star) = 2\sqrt{3} > 0$), giving linear convergence.

In [ ]:
def gd(x0, eta, steps):
    """Vanilla GD with subgradient selection at the kink. Returns array of iterates."""
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * subgrad(xs[-1]))
    return np.array(xs)

def sgd(x0, steps, eta0, gamma=None, sigma=SIGMA, rng=None):
    """SGD: x_{k+1} = x_k - eta_k (f'(x_k) + eps_k), eps_k ~ N(0, sigma^2).
    gamma=None -> constant step; else eta_k = eta0 / (1 + gamma k)."""
    rng = rng or np.random.default_rng(SEED)
    xs = [float(x0)]
    for k in range(steps):
        eta_k = eta0 if gamma is None else eta0 / (1.0 + gamma * k)
        noise = rng.normal(0.0, sigma)
        xs.append(xs[-1] - eta_k * (subgrad(xs[-1]) + noise))
    return np.array(xs)

def metrics(xs, tol=TOL):
    """final gap, best-so-far gap, steps-to-tolerance (np.nan if never reached)."""
    gaps = f(xs) - F_STAR
    hit = np.nonzero(gaps <= tol)[0]
    return {
        "final_gap": float(gaps[-1]),
        "best_gap": float(gaps.min()),
        "steps_to_tol": int(hit[0]) if hit.size else np.nan,
    }


### 6.1 Local Step Geometry

At $x_0 = 2$: $f'(x_0) = \tfrac12$, so with $\eta = 0.2$ the update is
$x' = x_0 - \eta f'(x_0) = 1.9$. The tangent line
$y = f(x_0) + f'(x_0)(x - x_0)$ shows how the local slope determines direction and length.

In [ ]:
x0_geom = 2.0
slope = fprime(x0_geom)
x_new = x0_geom - ETA_GEOM * slope
assert np.isclose(x_new, 1.9), "step-geometry sanity check (handout value)"

xs_loc = np.linspace(1.2, 2.6, 200)
tangent = f(x0_geom) + slope * (xs_loc - x0_geom)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs_loc, f(xs_loc), label="$f(x)$")
ax.plot(xs_loc, tangent, "--", c="#1b998b", label="tangent at $x_0$")
ax.plot([x0_geom], [f(x0_geom)], "o", c="k", label="$x_0 = 2$")
ax.plot([x_new], [f(x0_geom) + slope * (x_new - x0_geom)], "o", c="#e84855",
        label=f"tangent prediction at $x'={x_new:.1f}$")
ax.plot([x_new], [f(x_new)], "s", c="#5f6caf", ms=6,
        label=f"true $f(x') = {f(x_new):.4f}$")
ax.annotate("", xy=(x_new, f(x0_geom)), xytext=(x0_geom, f(x0_geom)),
            arrowprops=dict(arrowstyle="->", color="#e84855"))
ax.set(title=f"Gradient step at $x_0=2$, $\\eta={ETA_GEOM}$",
       xlabel="$x$", ylabel="$y$")
ax.legend()
plt.show()


### 6.2 Three Trajectories, Constant Step

$\eta = 0.15$ from $x_0 \in \{-1.0,\ 0.5,\ 2.0\}$; all should converge to
$x_\star \approx -0.1547$.

**Interpretation.** All three protocol starts reach $x_\star$: $x_0 = -1.0$ descends the
left slope directly, while $x_0 = 0.5$ and $x_0 = 2.0$ ride the interior branch leftward —
the longer the ride, the more iterations (quantified in §6.4).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
xs_plot = np.linspace(-1.2, 3.0, 600)
ax.plot(xs_plot, f(xs_plot), c="0.6", lw=1.2, label="$f(x)$")
for x0, c, z in zip(X0_LIST, ["#1b998b", "#e84855", "#5f6caf"], [3, 5, 4]):
    path = gd(x0, ETA_TRAJ, 60)   # zorder puts x0=0.5 on top (it shares the x0=2.0 corridor)
    ax.plot(path, f(path), "o-", ms=3, c=c, zorder=z, label=f"$x_0={x0}$")
ax.axvline(X_STAR, ls=":", c="k", lw=0.8)
ax.set(title=f"GD iterates, $\\eta={ETA_TRAJ}$", xlabel="$x$", ylabel="$f(x)$")
ax.legend()
plt.show()


### 6.3 Overshooting and Oscillation

From $x_0=0.5$ with $\eta=0.6$, the handout reports the sequence
$0.5,\ -0.475,\ 0.283,\ -0.454,\ 0.249,\ \dots$ alternating around $x_\star$.
The cell checks agreement with those rounded values to the specified tolerance —
a concrete correctness check against the source.

In [ ]:
seq = gd(0.5, ETA_OVERSHOOT, 4)
print("iterates:", np.round(seq, 3))
expected = np.array([0.5, -0.475, 0.283, -0.454, 0.249])
assert np.allclose(np.round(seq, 3), expected, atol=1e-3), \
    "overshoot sequence deviates from handout values"
print("matches handout sequence PASSED")

small = gd(0.5, 0.10, 25)
large = gd(0.5, ETA_OVERSHOOT, 25)
fig, ax = plt.subplots(figsize=(7.5, 4))
xs_loc = np.linspace(-0.8, 0.6, 400)
ax.plot(xs_loc, f(xs_loc), c="0.6", lw=1.2, label="$f(x)$")
ax.plot(small, f(small), "o-", ms=3, label="small $\\eta=0.10$")
ax.plot(large, f(large), "o-", ms=3, label=f"large $\\eta={ETA_OVERSHOOT}$")
ax.axvline(X_STAR, ls=":", c="k", lw=0.8, label="$x_\\star$")
ax.set(title="Overshooting with a large step size", xlabel="$x$", ylabel="$f(x)$")
ax.legend()
plt.show()


### 6.4 Step-Size Sensitivity, Full Protocol

GD for every $(x_0, \eta)$ pair of the protocol, $K = 200$: final gap per step size,
plus convergence curves. The handout's Fig. 9 is schematic; these are actual results.

**Honest deviation from the handout's schematic Fig. 9:** in the actual runs, GD reaches
the analytic minimum to machine precision for *every* protocol $(x_0, \eta)$ pair, so the
*final gap* does not discriminate between step sizes at $K=200$. The informative metric is
**steps-to-tolerance**, plotted below instead; the convergence curves tell the same story.

**Interpretation.** For the tested step sizes, steps-to-tolerance from $x_0 = 2.0$
decrease as the step size increases (41/20/13/10). This is a comparison within the
tested settings, not a general rule that larger stable steps converge faster.
The quadratic baseline (§9) explains how the local contraction factor depends on
step size; it does not guarantee global convergence for this nonlinear objective.

In [ ]:
results_gd = {}
for x0 in X0_LIST:
    for eta in ETA_GD:
        xs_run = gd(x0, eta, K)
        results_gd[(x0, eta)] = {"path": xs_run, **metrics(xs_run)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
width = 0.25
for i, x0 in enumerate(X0_LIST):
    stt = [results_gd[(x0, eta)]["steps_to_tol"] for eta in ETA_GD]
    axes[0].bar(np.arange(len(ETA_GD)) + (i - 1) * width, stt, width, label=f"$x_0={x0}$")
axes[0].set(title=f"Steps to tolerance ($f(x_k)-f(x_\\star) \\leq$ {TOL})",
            xlabel="step size $\\eta$", ylabel="iterations",
            xticks=range(len(ETA_GD)), xticklabels=[str(e) for e in ETA_GD])
axes[0].legend()

EPS = np.finfo(np.float64).eps
for eta in ETA_GD:
    gaps = f(results_gd[(0.5, eta)]["path"]) - F_STAR
    axes[1].semilogy(np.maximum(gaps, EPS), label=f"$\\eta={eta}$")
axes[1].set(title="Convergence from $x_0=0.5$", xlabel="iteration $k$",
            ylabel="$f(x_k)-f(x_\\star)$")
axes[1].legend()
fig.tight_layout()
plt.show()


### 6.5 Kink Probe: Fixed-Step GD Near $x=3$

Deterministic GD with $\eta = 0.15$ from $x_0 \in \{2.5, 2.8, 3.5\}$, run for
$K=200$ iterations with no early stopping. This probes the behavior near the kink
for these specific starts and step size. Starting exactly at $x_0=3$ would instead
remain fixed under our zero-subgradient selection.

**Interpretation:** over the simulated horizon, these trajectories oscillate near
$x=3$ without reaching the global-minimum tolerance. The reported late-stage bands
describe their observed range; they alone do not establish a periodic orbit or
permanent trapping. This observation is specific to the tested settings and does
not establish trapping for noisy SGD.



In [ ]:
basin_starts = [2.5, 2.8, 3.5]
fig, ax = plt.subplots(figsize=(8, 4))
for x0 in basin_starts:
    path = gd(x0, 0.15, K)
    tail = path[-50:]
    hit = np.nonzero(f(path) - F_STAR <= TOL)[0]
    print(f"x0 = {x0}: reached tol = {bool(hit.size)}, "
          f"late-stage band = [{tail.min():.3f}, {tail.max():.3f}]")
    ax.plot(path, lw=1.0, label=f"$x_0 = {x0}$")
ax.axhline(3.0, ls="--", c="gray", lw=0.9, label="kink $x=3$")
ax.axhline(X_STAR, ls=":", c="k", lw=0.9, label="$x_\\star$")
ax.set(title="Fixed-step GD ($\\eta=0.15$) near the kink",
       xlabel="iteration $k$", ylabel="$x_k$")
ax.legend(fontsize=8)
plt.show()



## 7. Stochastic Gradient Descent

$$
x_{k+1} = x_k - \eta_k\big(g_k + \varepsilon_k\big), \qquad
\mathbb{E}[\varepsilon_k \mid x_k] = 0,\quad \mathbb{E}[\varepsilon_k^2 \mid x_k] \le \sigma^2 .
$$

Here $g_k=f'(x_k)$ for $x_k\ne 3$, and $g_k=0$ at $x_k=3$, matching our selected subgradient. Unlike deterministic GD, noisy SGD generally moves away from the kink even when initialized exactly there.

For the linearized model around $x_\star$, let $y_k=x_k-x_\star$ and
$\mu=f''(x_\star)=2\sqrt{3}$. With independent, zero-mean noise of variance
$\sigma^2$ and a constant step satisfying $0<\eta\mu<2$, the limiting second moment
(handout, Prop. 2) is

$$
\lim_{k\to\infty} \mathbb{E}\,y_k^2 \;=\; \frac{\eta\sigma^2}{2\mu - \eta\mu^2}
\;\approx\; \frac{\eta\sigma^2}{2\mu}\quad (\eta\mu \ll 1),
$$

Thus, for small $\eta\mu$, the linear model's limiting RMS distance scales like
$\sqrt{\eta}\,\sigma/\sqrt{2\mu}$. This describes convergence of its second moment,
not convergence of individual noisy trajectories to a fixed limit. For the nonlinear
objective, it is a local approximation.
Diminishing schedules $\eta_k=\eta_0/(1+\gamma k)$ with $\eta_0>0$ and $\gamma>0$
satisfy the Robbins–Monro step-size conditions, but those conditions alone do not
guarantee convergence to the global minimizer of this nonconvex objective.

### 7.1 Sample Paths

**Disclosure:** $x_0 = 2.2$ is the handout *Figure 7* illustration value, used here only
for that illustration. It lies to the right of the local maximizer $x_+$.
The three plotted paths show finite-horizon behavior with $\eta=0.12$ and
$\sigma=0.5$; they do not establish permanent trapping or eventual convergence.
The *protocol* experiments in §7.2 use the prescribed starts
$x_0 \in \{-1.0, 0.5, 2.0\}$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for i in range(3):
    path = sgd(2.2, 40, eta0=0.12, rng=np.random.default_rng(SEED + i))
    ax.plot(path, "o-", ms=2.5, label=f"path {chr(65 + i)} (seed {SEED + i})")
ax.axhline(X_STAR, ls=":", c="k", lw=0.9, label="$x_\\star$")
ax.set(title=f"SGD sample paths, constant $\\eta=0.12$, $\\sigma={SIGMA}$",
       xlabel="iteration $k$", ylabel="$x_k$")
ax.legend()
plt.show()


### 7.2 Constant vs Diminishing Schedules

Replicates handout Fig. 8 with real protocol runs. Full Section 7 protocol: starts $x_0 \in \{-1.0, 0.5, 2.0\}$, constant
$\eta_0 \in \{0.2, 0.1\}$ and diminishing $\eta_k = \eta_0/(1+\gamma k)$ with
$\gamma \in \{0.02, 0.05\}$, $K = 200$, **20 seeds per configuration**. The figure shows
the representative start $x_0 = 0.5$ as **median curves with interquartile bands**
(clipped at machine epsilon for the log scale — disclosed); the paired statistics and
metrics table below cover all three starts.

**Endpoint location:** the counts below report how many runs from $x_0=2.0$ end to the right of the local maximizer $x_+$.
Endpoint location alone does not establish whether a trajectory entered, remained near, or escaped from the kink region.

In [ ]:
N_REPS = 20                      # seeds per configuration
SGD_SEEDS = [SEED + 100 + r for r in range(N_REPS)]   # recorded
EPS64 = np.finfo(np.float64).eps

def gap_matrix(x0, eta0, gamma):
    """(N_REPS, K+1) matrix of objective gaps for one configuration."""
    rows = [f(sgd(x0, K, eta0=eta0, gamma=gamma, rng=np.random.default_rng(s))) - F_STAR
            for s in SGD_SEEDS]
    return np.vstack(rows)

# Full protocol grid: 3 starts x 2 eta0 x {const, 0.02, 0.05}
results_sgd = {}
for x0 in X0_LIST:
    for eta0 in ETA0_SGD:
        for gamma in [None] + GAMMA_SGD:
            results_sgd[(x0, eta0, gamma)] = gap_matrix(x0, eta0, gamma)

# Figure: representative start x0 = 0.5, median with IQR bands
X0_FIG = 0.5
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for eta0, base_c in zip(ETA0_SGD, ["#1b998b", "#5f6caf"]):
    for gamma, ls in [(None, "-"), (0.02, "--"), (0.05, ":")]:
        G = np.maximum(results_sgd[(X0_FIG, eta0, gamma)], EPS64)
        med = np.median(G, axis=0)
        q1, q3 = np.percentile(G, [25, 75], axis=0)
        lab = (f"constant $\\eta={eta0}$" if gamma is None
               else f"$\\eta_k={eta0}/(1+{gamma}k)$")
        ax.semilogy(med, ls, c=base_c, label=lab)
        ax.fill_between(range(K + 1), q1, q3, color=base_c, alpha=0.12)
ax.set(title=f"Objective gap from $x_0={X0_FIG}$: median of {N_REPS} seeds, IQR shaded "
             f"(gaps clipped at machine eps for log scale)",
       xlabel="iteration $k$", ylabel="$f(x_k)-f(x_\\star)$")
ax.legend(fontsize=8, loc="upper right")
plt.show()

# Paired statistics: diminishing vs constant (same eta0, same seed), final gaps
print("Paired final-gap comparison (diminishing vs constant, same seed):")
for x0 in X0_LIST:
    for eta0 in ETA0_SGD:
        fc = results_sgd[(x0, eta0, None)][:, -1]
        for gamma in GAMMA_SGD:
            fd = results_sgd[(x0, eta0, gamma)][:, -1]
            wins = int((fd < fc).sum())
            ratio = float(np.median(fd / np.maximum(fc, EPS64)))
            print(f"  x0={x0:5}, eta0={eta0}, gamma={gamma}: "
                  f"diminishing wins {wins}/{N_REPS}, median ratio {ratio:.3f}")

# Endpoint counts for x0 = 2.0: final iterate right of the local maximizer
print("\nEndpoint counts (x0 = 2.0): seeds ending with x_K > x_+")
for eta0 in ETA0_SGD:
    for gamma in [None] + GAMMA_SGD:
        xK = np.array([sgd(2.0, K, eta0=eta0, gamma=gamma,
                           rng=np.random.default_rng(s))[-1] for s in SGD_SEEDS])
        trapped = int((xK > X_PLUS).sum())
        gname = "const" if gamma is None else f"g={gamma}"
        print(f"  eta0={eta0}, {gname}: right of x_+ {trapped}/{N_REPS}")


## 8. Metrics Summary

GD rows summarize individual deterministic runs. SGD rows summarize metrics calculated separately for each seed from raw, unclipped objective gaps. Gap and first-hit summaries show the median [Q1, Q3]; first-hit statistics include only successful runs. Hits include iteration zero, and non-hits are reported separately. First crossing does not imply remaining within tolerance.

The constant-step comparison uses the representative start x0=0.5. Monte Carlo standard errors describe uncertainty in the empirical mean across independent seeds. Predictions refer to the linearized model's expected quadratic gap, at the simulated horizon and in its stationary limit; the empirical quantity is the nonlinear objective gap. Agreement is approximate and does not establish nonlinear stationarity.

In [ ]:
header = f"{'method':<26}{'x0':>6}{'eta':>12}{'final gap':>12}{'best gap':>12}{'steps<tol':>11}"
print(header); print("-" * len(header))
for (x0, eta), res in results_gd.items():
    print(f"{'GD constant':<26}{x0:>6}{eta:>12}{res['final_gap']:>12.2e}"
          f"{res['best_gap']:>12.2e}{str(res['steps_to_tol']):>11}")
from IPython.display import HTML, display

def html_summary_parts(summary):
    if summary == "n/a":
        return "n/a", ""
    median, quartiles = summary.split(" ", 1)
    return median, quartiles

sgd_rows = []
for (x0, eta0, gamma), G in results_sgd.items():
    label = "constant" if gamma is None else f"diminishing g={gamma}"
    total = G.shape[0]
    final_gap = G[:, -1]
    best_gap = G.min(axis=1)
    hit_mask = np.any(G <= TOL, axis=1)
    first_hit = np.full(total, -1, dtype=int)
    for row in np.flatnonzero(hit_mask):
        first_hit[row] = int(np.flatnonzero(G[row] <= TOL)[0])
    successful_hits = first_hit[hit_mask]

    def gap_stats(values):
        q1, med, q3 = np.percentile(values, [25, 50, 75])
        return f"{med:.2e} [{q1:.2e},{q3:.2e}]"

    def iter_stats(values):
        if values.size == 0:
            return "n/a"
        q1, med, q3 = np.percentile(values, [25, 50, 75])
        return f"{med:.2f} [{q1:.2f},{q3:.2f}]"

    hits = int(hit_mask.sum())
    final_median, final_quartiles = html_summary_parts(gap_stats(final_gap))
    best_median, best_quartiles = html_summary_parts(gap_stats(best_gap))
    first_hit_median, first_hit_quartiles = html_summary_parts(iter_stats(successful_hits))
    final_cell = f"{final_median}<br><span class='wk01-muted'>{final_quartiles}</span>"
    best_cell = f"{best_median}<br><span class='wk01-muted'>{best_quartiles}</span>"
    first_hit_cell = (f"{first_hit_median}<br><span class='wk01-muted'>{first_hit_quartiles}</span>"
                      if first_hit_quartiles else first_hit_median)
    sgd_rows.append(
        f"<tr><td>{label}</td><td class='wk01-num'>{x0:.1f}</td><td class='wk01-num'>{eta0:.2f}</td>"
        f"<td class='wk01-num'>{final_cell}</td><td class='wk01-num'>{best_cell}</td>"
        f"<td class='wk01-num'>{hits}/{total} ({100 * hits / total:.0f}%)</td>"
        f"<td class='wk01-num'>{total - hits}</td><td class='wk01-num'>{first_hit_cell}</td></tr>"
    )

display(HTML("""
<style>
.wk01-sgd-table, .wk01-theory-table { border-collapse: collapse; margin: .5rem 0 1rem; font-size: .92rem; }
.wk01-sgd-table th, .wk01-sgd-table td, .wk01-theory-table th, .wk01-theory-table td { border-bottom: 1px solid #d9dee5; padding: .35rem .55rem; vertical-align: top; }
.wk01-sgd-table th, .wk01-theory-table th { background: #f4f6f8; font-weight: 600; text-align: left; white-space: nowrap; }
.wk01-sgd-table .wk01-num, .wk01-theory-table .wk01-num { text-align: right; font-variant-numeric: tabular-nums; white-space: nowrap; }
.wk01-muted { color: #5f6b7a; font-size: .9em; }
</style>
<table class='wk01-sgd-table'>
<caption><strong>SGD endpoint summary</strong>; first-hit statistics are conditional on success</caption>
<thead><tr><th>Configuration</th><th>Start</th><th>η</th><th>Final gap<br><span class='wk01-muted'>median / [Q1, Q3]</span></th><th>Best gap<br><span class='wk01-muted'>median / [Q1, Q3]</span></th><th>Hits</th><th>Non-hits</th><th>First hit<br><span class='wk01-muted'>median / [Q1, Q3]</span></th></tr></thead>
<tbody>""" + "".join(sgd_rows) + """</tbody></table>
"""))

theory_rows = []
mu = 2 * np.sqrt(3)
y0 = 0.5 - X_STAR
for eta0 in ETA0_SGD:
    if not 0 < eta0 * mu < 2:
        raise ValueError(f"linearized formula requires 0 < eta*mu < 2, got {eta0 * mu}")
    G = results_sgd[(0.5, eta0, None)]
    endpoint_gaps = G[:, -1]
    empirical_mean = endpoint_gaps.mean()
    se = endpoint_gaps.std(ddof=1) / np.sqrt(G.shape[0])
    r = 1 - eta0 * mu
    m_K = r**(2 * K) * y0**2 + eta0**2 * SIGMA**2 * (1 - r**(2 * K)) / (1 - r**2)
    linear_at_K = 0.5 * mu * m_K
    stationary = eta0 * SIGMA**2 / (2 * (2 - eta0 * mu))
    theory_rows.append(f"<tr><td class='wk01-num'>{eta0:.2f}</td><td class='wk01-num'>{empirical_mean:.3e}</td><td class='wk01-num'>{se:.3e}</td><td class='wk01-num'>{linear_at_K:.3e}</td><td class='wk01-num'>{stationary:.3e}</td></tr>")

display(HTML("""
<table class='wk01-theory-table'>
<caption><strong>Constant-step SGD mean vs linearized theory</strong> (x0=0.5)</caption>
<thead><tr><th>η</th><th>Empirical mean gap</th><th>Monte Carlo SE</th><th>Linear model at K</th><th>Stationary linear model</th></tr></thead>
<tbody>""" + "".join(theory_rows) + """</tbody></table>
"""))


## 9. Quadratic Baseline — Stability Reference

For the quadratic baseline $q(x)=a x^2/2$ with $a=2\sqrt{3}$, the GD update is exactly $x_{k+1}=(1-\eta a)x_k$. Convergence to zero from every initial point holds when $0<\eta a<2$, with contraction factor $|1-\eta a|$. For the nonlinear objective, the matching curvature at $x_\star$ gives a local attraction condition for sufficiently nearby starts, not a global convergence guarantee. The one-sided curvature at the kink does not provide a stability bound for the nonsmooth update. The cell checks the quadratic contraction factor and illustrates divergence just above $2/a$.

In [ ]:
a = 2.0 * np.sqrt(3.0)                       # local curvature at x_star
def q(x): return 0.5 * a * x**2
def qgrad(x): return a * x

print(f"quadratic stability interval: 0 < eta < 2/a = {2/a:.4f}")
for eta in [0.10, 0.20, 0.50, 0.60]:
    x = 1.0
    ratios = []
    for _ in range(30):
        x_new = x - eta * qgrad(x)
        if abs(x) > 0:
            ratios.append(abs(x_new) / abs(x))
        x = x_new
        if abs(x) > 1e6:
            break
    emp = max(ratios)
    theory = abs(1 - eta * a)
    status = "stable" if eta < 2 / a else "DIVERGES"
    print(f"eta={eta:.2f}: |1-eta*a|={theory:.3f}  empirical max ratio={emp:.3f}  -> {status}")
    if eta < 2 / a:
        assert np.isclose(emp, theory, atol=1e-9), "contraction mismatch"
print("quadratic contraction check PASSED")

## 10. Interpretation Notes

- **Calculus → algorithm.** The sign structure of $f'$ funnels all three protocol starts
  to $x_\star$: every GD run reaches the $10^{-3}$ tolerance in 2–41 steps and machine
  precision ($\sim 10^{-16}$) well before $K = 200$.
- **Step sizes.** Larger tested steps reach tolerance sooner from the protocol starts, as summarized in §6.4. This empirical ordering is specific to these settings. At the overshoot demonstration's step size, the quadratic baseline diverges; for the nonlinear objective, the same local calculation establishes instability of the smooth minimizer, not divergence of every trajectory.
- **SGD noise comparison.** Section 8 compares empirical mean endpoint gaps and Monte Carlo standard errors with finite-horizon and stationary predictions from the linearized model. Comparing an empirical median with a theoretical expectation is inappropriate; this comparison uses means and remains subject to local-linearization and finite-sample limitations.
- **Constant vs diminishing.** Diminishing achieves lower final gaps **by $K = 200$** in
  16–19 of 20 paired seeds (median ratios 0.044–0.142), trading slower early progress;
  the median is not monotone, so this is not a claim of steady convergence.
- **Shared noise across starts.** Reusing each seed pairs the noise sequences across starting points for a given schedule. This can make comparisons more informative, but it does not guarantee identical trajectories or endpoints. Runs sharing a noise sequence should not be pooled as independent observations.
- **Kink behavior.** For the starts and step size tested in §6.5, deterministic GD oscillates near the kink over the simulated horizon without reaching the global-minimum tolerance. The SGD counts in §7.2 describe endpoint location only; they do not establish entry, trapping, or escape histories.
- **Reading the figures.** Section 6.4 reports steps-to-tolerance because final GD gaps reach floating-point precision in the tested runs. Figure 7 shows three finite-horizon SGD paths from the illustration start; these paths alone do not establish long-run convergence or trapping.

## 11. Limitations & Extensions

Not implemented (handout §9 marks these as extensions, not requirements): heavy-ball /
Nesterov comparison, Adam, contraction-bound plot (Fig. 10), proximal step for
$f(x)+\lambda|x|$, multi-dimensional separable variants, and smoothed
$\sqrt{g^2+\delta^2}$ approximations. Kink handling uses the smallest-norm subgradient;
alternatives (backtracking across the kink) are noted in handout §8.

